# **DATA SCIENCES PYTHON PROJECT - TOXIC COMMENT FILTERING**

# Project Overview:
### - Develop and demonstrate a multi-label toxicity classifier for a gaming platform’s live chat. 
### - The system flags messages as spam, insult, profanity, -- hate speech, or non-toxic, assigns a 0-5 severity score,
### - and outputs recommended moderation actions.


## **WHAT IS TOXIC COMMENT FILTERING ?**
### - Toxic comment filtering = catching bad, hurtful, abusive messages and stopping them.

# **BUSINESS REQUIREMENTS**
### 1.Detect toxic comments
### 2.Give severity score
### 3.Take moderation actions
### 4.Work in real-time chat

# **IBHWSS Formula**
## **I → Idea (What is Exaclty Project?)**

### This project identifies toxic comments in chat messages and decides whether the message should be allowed, warned, muted, or auto-banned.

## **B → Building (What are we building?)**
### We are building a **multi-label toxicity** detection system that:
### Reads a comment
### leans and preprocesses the text
### Predicts multiple toxic categories at the same time
### Calculates severity (0–5)
### Suggests moderation action

## **H → How will the program work?**
### Load dataset containing comments + labels
### Clean & preprocess text
### Convert text to **TF-IDF vectors**
### Train model using **MultiOutputClassifier(LogisticRegression)**
### Predict toxicity labels for new comments
### Generate severity score (sum of predicted labels)
### Output moderation action

## **W → Which concept/model are we using?**
### Concepts: **NLP preprocessing, TF-IDF, multi-label classification**
### Model: **MultiOutputClassifier (with LinearSVC inside)**
### Format: Predict 6 toxic labels → sum them → assign severity → decide action

## **S → Steps of pseudocode**
### a) Understand the problem
### We must detect multiple toxic behaviors (toxic, insult, threat, etc.) from a single comment.
### b) Break into smaller steps with solution
### Load dataset
### Clean the text
### Extract features using TF-IDF
### Split train/test
### Train **MultiOutputClassifier**
### Predict labels
### Compute severity score
### Decide moderation action

### c) Pseudocode
### LOAD dataset
### CLEAN comment_text column
### X = clean_text
### y = LABELS
### SPLIT X, y into train/test
### **TFIDF = fit_transform(X_train)**
### **MODEL = MultiOutputClassifier(LogisticRegression).fit()**
### PRED_PROBA = MODEL.predict_proba(X_test)
### BINARY_PRED = convert probabilities to 0/1
### SEVERITY = sum of 1's
### ACTION = map severity to moderation rule
### SAVE TFIDF + MODEL
### d) Test pseudocode
### Test with small comments
### “Hello friend” → non-toxic
### “You are stupid” → toxic / insult
### “Go die” → threat / severe toxic
### Check outputs and severity.
### e) Translate to actual code
### **Use Python + pandas + sklearn:**
### **MultiOutputClassifier(LogisticRegression())**
### **TfidfVectorizer**
### train_test_split
### prediction + evaluation code

## **S → Stretching (Next-level ideas)**
### a]Add sentiment analysis
### b]Deploy as API
### c]Add real-time chat monitoring
### d]Use BERT later for higher accuracy
### e]Tune thresholds per label

# **Step 1 Import Libraries** 

In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Text preprocessing
import re
import string

# Feature extraction
from sklearn.feature_extraction.text import TfidfVectorizer
# Model and multi-label handling
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, multilabel_confusion_matrix
from scipy.sparse import hstack

# Saving model
import joblib

# **Step 2 Load Dataset** 

In [2]:
data = pd.read_csv("train.csv")

# **Step 3 Exploratory Data Analysis(EDA)** 
## Quickly Check Your Data How Look Like

In [3]:
# Check all this print(data.sha
data.describe()
data.isnull().sum()  # Check for missing values
data.head()
data.tail()
data.describe()
data.shape

(159571, 8)

# **Text Preprocessing**

In [4]:
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)
    text = re.sub(r"@\w+", '', text)
    text = re.sub(r"[^\w\s]", '', text)
    text = re.sub(r"\s+", ' ', text).strip()
    return text

data['clean_text'] = data['comment_text'].apply(preprocess)

In [5]:
print(data[['comment_text', 'clean_text']].head())

                                        comment_text  \
0  Explanation\nWhy the edits made under my usern...   
1  D'aww! He matches this background colour I'm s...   
2  Hey man, I'm really not trying to edit war. It...   
3  "\nMore\nI can't make any real suggestions on ...   
4  You, sir, are my hero. Any chance you remember...   

                                          clean_text  
0  explanation why the edits made under my userna...  
1  daww he matches this background colour im seem...  
2  hey man im really not trying to edit war its j...  
3  more i cant make any real suggestions on impro...  
4  you sir are my hero any chance you remember wh...  


In [6]:
# X, y Features
labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
X = data['clean_text']
y = data[labels]

In [7]:
data.columns

Index(['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat',
       'insult', 'identity_hate', 'clean_text'],
      dtype='object')

# **Train-Test-Split**

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# **Feature Engineering – TF-IDF**

In [9]:
vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1,3))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [10]:
severe_keywords = ['kill','die','rape','hang','burn','lynch','bomb','murder','shoot','gas']
X_train_flag = X_train.apply(lambda x: int(any(word in x.split() for word in severe_keywords)))
X_test_flag = X_test.apply(lambda x: int(any(word in x.split() for word in severe_keywords)))

In [11]:
# Message length
X_train_len = X_train.apply(lambda x: len(x.split()))
X_test_len  = X_test.apply(lambda x: len(x.split()))

# Exclamation count
X_train_excl = X_train.apply(lambda x: x.count('!'))
X_test_excl  = X_test.apply(lambda x: x.count('!'))

In [12]:
print("Data preparation done")
print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

Data preparation done
Train shape: (127656,) (127656, 6)
Test shape: (31915,) (31915, 6)


In [14]:
X_train_final = hstack([X_train_tfidf, np.array(X_train_flag).reshape(-1,1)])
X_test_final = hstack([X_test_tfidf, np.array(X_test_flag).reshape(-1,1)])

# **Choose Model Train**

In [15]:
# Train model
clf = LogisticRegression(max_iter=2000, class_weight={0:1, 1:10})
multi_clf = MultiOutputClassifier(clf)
multi_clf.fit(X_train_final, y_train)

MultiOutputClassifier(estimator=LogisticRegression(class_weight={0: 1, 1: 10},
                                                   max_iter=2000))

# **Make Prediction**

In [17]:
# Create empty predictions DataFrame with same index as y_test
y_pred = pd.DataFrame(index=y_test.index)

# Probabilities
y_prob_severe = multi_clf.estimators_[1].predict_proba(X_test_final)[:, 1]

# Threshold
threshold = 0.75
y_pred['severe_toxic'] = (y_prob_severe >= threshold).astype(int)

# **Evaluation**

In [20]:
# Evaluate all labels
for label in labels:
    print(f"\n{label.upper()}")
    print(classification_report(y_test[label], y_pred[label]))


TOXIC
              precision    recall  f1-score   support

           0       0.98      0.94      0.96     28859
           1       0.61      0.85      0.71      3056

    accuracy                           0.93     31915
   macro avg       0.80      0.90      0.84     31915
weighted avg       0.95      0.93      0.94     31915


SEVERE_TOXIC
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     31594
           1       0.47      0.49      0.48       321

    accuracy                           0.99     31915
   macro avg       0.73      0.74      0.74     31915
weighted avg       0.99      0.99      0.99     31915


OBSCENE
              precision    recall  f1-score   support

           0       0.99      0.98      0.99     30200
           1       0.72      0.84      0.77      1715

    accuracy                           0.97     31915
   macro avg       0.85      0.91      0.88     31915
weighted avg       0.98      0.97      0.97 

In [21]:
joblib.dump(multi_clf, 'multi_label_toxicity_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

['tfidf_vectorizer.pkl']

# **Detect toxic comments**

In [22]:
def detect_toxic(comment):
    comment_clean = preprocess_text(comment)
    comment_tf = tfidf.transform([comment_clean])
    pred = clf.predic(comment_tf)[0]
    return pred # 1 = toxic, 0 = safe

# **Give severity score**

In [23]:
def detect_toxic(comment):
    # 1. Clean the comment using your actual cleaning function
    comment_clean = clean_text(comment)   # <-- replace preprocess_text with clean_text
    
    # 2. Transform text using trained TF-IDF vectorizer
    comment_tf = tfidf.transform([comment_clean])
    
    # 3. Predict using your trained model (fix typo: predic -> predict)
    pred = clf.predict(comment_tf)[0]
    
    # 4. Return predictions as a dictionary with label names
    return dict(zip(y_test.columns, pred))


# **Take moderation actions**

In [24]:
# Example dataset
comments = [
    "You are amazing!",
    "I hate you",
    "You are horrible",
    "Great job",
    "You are stupid",
    "Well done"
]
labels = [0, 1, 1, 0, 1, 0]  # 1 = toxic, 0 = non-toxic

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(comments, labels, test_size=0.3, random_state=42)

#Train TF-IDF vectorizer
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Train a simple classifier
clf = LogisticRegression()
clf.fit(X_train_tfidf, y_train)

# Text cleaning function
def clean_text(text):
    text = str(text).lower()
    return text

# Detect toxic function
def detect_toxic(comment):
    comment_clean = clean_text(comment)
    comment_tf = tfidf.transform([comment_clean])
    pred = clf.predict(comment_tf)[0]
    return pred

# Test 
comment = "You are a horrible person!"
preds = detect_toxic(comment)
print("Predicted toxic label:", preds)

Predicted toxic label: 1


# **Work in real-time chat**

In [ ]:
while True:
    user_input = input("User: ")
    if user_input.lower() in ["exit", "quit"]:
        break
    preds = detect_toxic(user_input)
    score = severity_score(preds)
    action = moderation_action_with_score(score)
    print(f"Severity Score: {score} → Action: {action}")

# **Conclusion**
### A multi-label toxicity classification system was developed using the Jigsaw Toxic Comment dataset to analyze live gaming chat messages. 
### The model effectively identifies multiple categories of toxic content and assigns a severity score to support moderation decisions.
### While overall performance is strong across most labels, the Severe Toxic class did not fully reach the target precision of 80% due to class imbalance. 
### This limitation highlights an area for future improvement through further threshold tuning or advanced models.